<a href="https://colab.research.google.com/github/ver1812/Capstone_Project/blob/main/results/results_consolidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-PIDS: Results Consolidation

Consolidation of all results.

**Phase 1: Original exploration (all models tried, before narrowing scope):**
LogisticRegression, LinearSVC, BiLSTM, BERT-base, DistilBERT, ModernBERT — Accuracy/F1/Precision/Recall.

**Phase 2: Detailed analysis on the 3 finalized models** (LinearSVC, DistilBERT, ModernBERT):
Retested results (Accuracy/F1/Precision/Recall/FN/FP,ROC_AUC).

**Phase 3 — Best model (ModernBERT) deep dive:**
Zero-shot generalization gap (test_v2 vs. BIPIA held-out) and Phase 2 fine-tuning results.



## 1. Imports

In [1]:
import json
import os

import pandas as pd

print("Imports ready.")


Imports ready.


## 2. Mount Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 3. Paths



In [3]:
BASE_DIR = "/content/drive/MyDrive/Capstone"
SAVED_DIR = os.path.join(BASE_DIR, "saved")
EVAL_RESULTS_DIR = os.path.join(BASE_DIR, "eval", "results")

# Phase 1 — original per-model results files
TRADITIONAL_RESULTS_PATH = os.path.join(SAVED_DIR, "traditional_ml_v2", "traditional_ml_results.json")
BILSTM_RESULTS_PATH = os.path.join(SAVED_DIR, "glove_300d_1024", "bilstm_glove300_maxlen1024_v2_results.json")
BILSTM_100_512_RESULTS_PATH = os.path.join(SAVED_DIR, "glove_100_512", "bilstm_glove100_maxlen512_v2_results.json")
BILSTM_100_1024_RESULTS_PATH = os.path.join(SAVED_DIR, "glove_100d_1024", "bilstm_glove100_maxlen1024_v2_results.json")
BERT_RESULTS_PATH = os.path.join(SAVED_DIR, "bert", "bert_results.json")
DISTILBERT_RESULTS_PATH = os.path.join(SAVED_DIR, "distilbert", "distilbert_results.json")
MODERNBERT_RESULTS_PATH = os.path.join(SAVED_DIR, "modernbert", "modernbert_results.json")

# Phase 2 — fresh eval-notebook results
BASELINE_RESULTS_PATH = os.path.join(EVAL_RESULTS_DIR, "baseline_results_v2.json")
INFERENCE_SPEED_RESULTS_PATH = os.path.join(EVAL_RESULTS_DIR, "inference_speed_results_v2.json")

# Phase 3 — Phase 2 fine-tuning results
FINETUNED_RESULTS_PATH = os.path.join(SAVED_DIR, "modernbert_bipia_finetuned", "modernbert_bipia_finetuned_results.json")

print("Paths configured.")


Paths configured.


## 4. Helper functions

In [8]:
def load_json(path):
    with open(path) as f:
        return json.load(f)


def extract_row(model_label, results_dict, include_fn_fp):
    test_metrics = results_dict.get("test", {})
    row = {
        "model": model_label,
        "accuracy": test_metrics.get("accuracy"),
        "f1": test_metrics.get("f1"),
        "precision": test_metrics.get("precision"),
        "recall": test_metrics.get("recall"),
    }
    if include_fn_fp:
        confusion_matrix = results_dict.get("test_confusion_matrix", {})
        row["roc_auc"] = test_metrics.get("roc_auc")
        row["fn"] = confusion_matrix.get("fn")
        row["fp"] = confusion_matrix.get("fp")
    return row


def extract_zero_shot_bipia_row(model_label, zero_shot_dict):
    confusion_matrix = zero_shot_dict.get("confusion_matrix", {})
    return {
        "model": model_label,
        "accuracy": zero_shot_dict.get("accuracy"),
        "f1": zero_shot_dict.get("f1"),
        "precision": zero_shot_dict.get("precision"),
        "recall": zero_shot_dict.get("recall"),
        "roc_auc": zero_shot_dict.get("roc_auc"),
        "fn": confusion_matrix.get("fn"),
        "fp": confusion_matrix.get("fp"),
    }

## 5. Phase 1: Original results, all models

In [9]:
traditional_results = load_json(TRADITIONAL_RESULTS_PATH)
bilstm_300_1024_results = load_json(BILSTM_RESULTS_PATH)
bilstm_100_512_results = load_json(BILSTM_100_512_RESULTS_PATH)
bilstm_100_1024_results = load_json(BILSTM_100_1024_RESULTS_PATH)
bert_results = load_json(BERT_RESULTS_PATH)
distilbert_original_results = load_json(DISTILBERT_RESULTS_PATH)
modernbert_original_results = load_json(MODERNBERT_RESULTS_PATH)

phase1_rows = [
    extract_row("LogisticRegression", traditional_results["logistic_regression"], include_fn_fp=False),
    extract_row("LinearSVC", traditional_results["linear_svm"], include_fn_fp=False),
    extract_row("BiLSTM (GloVe 100d/512)", bilstm_100_512_results["bilstm"], include_fn_fp=False),
    extract_row("BiLSTM (GloVe 100d/1024)", bilstm_100_1024_results["bilstm"], include_fn_fp=False),
    extract_row("BiLSTM (GloVe 300d/1024)", bilstm_300_1024_results["bilstm"], include_fn_fp=False),
    extract_row("BERT-base", bert_results["bert"], include_fn_fp=False),
    extract_row("DistilBERT", distilbert_original_results["distilbert"], include_fn_fp=False),
    extract_row("ModernBERT", modernbert_original_results["modernbert"], include_fn_fp=False),
]

phase1_df = pd.DataFrame(phase1_rows).set_index("model")
phase1_df

,accuracy,f1,precision,recall
model,,,,
LogisticRegression,0.912706,0.907407,0.921309,0.893919
LinearSVC,0.914969,0.909839,0.923452,0.896622
BiLSTM (GloVe 100d/512),0.901390,0.895441,0.908838,0.882432
BiLSTM (GloVe 100d/1024),0.897187,0.889813,0.913229,0.867568
BiLSTM (GloVe 300d/1024),0.907856,0.902028,0.918125,0.886486
BERT-base,0.933398,0.927821,0.963610,0.894595
DistilBERT,0.928872,0.924190,0.943038,0.906081
ModernBERT,0.951180,0.947697,0.972281,0.924324


## 6. Phase 2a: Test results, 3 finalized models



In [10]:
baseline_results = load_json(BASELINE_RESULTS_PATH)

phase2_test_rows = [
    extract_row("LinearSVC", baseline_results["linear_svm"], include_fn_fp=True),
    extract_row("DistilBERT", baseline_results["distilbert"], include_fn_fp=True),
    extract_row("ModernBERT", baseline_results["modernbert"], include_fn_fp=True),
]

phase2_test_df = pd.DataFrame(phase2_test_rows).set_index("model")
phase2_test_df


,accuracy,f1,precision,recall,roc_auc,fn,fp
model,,,,,,,
LinearSVC,0.914969,0.909839,0.923452,0.896622,0.960035,153,110
DistilBERT,0.928548,0.923872,0.942375,0.906081,0.972625,139,82
ModernBERT,0.950857,0.947332,0.972262,0.923649,0.985400,113,39


## 7. Phase 2b: Inference latency, 3 finalized models

In [11]:
inference_speed_results = load_json(INFERENCE_SPEED_RESULTS_PATH)
speed_summary = inference_speed_results["summary"]

phase2_time_rows = []
for model_key, model_label in [("linear_svm", "LinearSVC"), ("distilbert", "DistilBERT"), ("modernbert", "ModernBERT")]:
    stats = speed_summary[model_key]
    phase2_time_rows.append({
        "model": model_label,
        "mean_ms": stats["mean_ms"],
        "median_ms": stats["median_ms"],
        "std_ms": stats["std_ms"],
        "min_ms": stats["min_ms"],
        "max_ms": stats["max_ms"],
    })

phase2_time_df = pd.DataFrame(phase2_time_rows).set_index("model")
phase2_time_df


,mean_ms,median_ms,std_ms,min_ms,max_ms
model,,,,,
LinearSVC,1.035978,1.028417,0.198776,0.802655,1.675530
DistilBERT,7.613932,5.205543,10.929679,4.771056,73.319688
ModernBERT,20.206380,20.040248,2.470972,17.460245,32.111879


## 8. Phase 3: Best model (ModernBERT) deep dive

Generalization gap (zero-shot) and Phase 2 fine-tuning results

In [12]:
finetuned_results = load_json(FINETUNED_RESULTS_PATH)

phase3_rows = [
    extract_row("ModernBERT zero-shot — test_v2 (direct)", modernbert_original_results["modernbert"], include_fn_fp=True),
    extract_zero_shot_bipia_row("ModernBERT zero-shot — BIPIA (indirect)", finetuned_results["zero_shot_bipia_reference"]),
    extract_row("ModernBERT fine-tuned — test_v2 (direct)", finetuned_results["test_v2_direct"], include_fn_fp=True),
    extract_row("ModernBERT fine-tuned — BIPIA (indirect)", finetuned_results["bipia_held_out"], include_fn_fp=True),
    extract_row("ModernBERT fine-tuned — combined", finetuned_results["combined_test_v2_plus_bipia"], include_fn_fp=True),
]

phase3_df = pd.DataFrame(phase3_rows).set_index("model")
phase3_df


,accuracy,f1,precision,recall,roc_auc,fn,fp
model,,,,,,,
ModernBERT zero-shot — test_v2 (direct),0.951180,0.947697,0.972281,0.924324,0.985409,112,39
ModernBERT zero-shot — BIPIA (indirect),0.560900,0.469100,0.593100,0.388000,0.613100,3060,1331
ModernBERT fine-tuned — test_v2 (direct),0.938894,0.936853,0.926636,0.947297,0.984913,78,111
ModernBERT fine-tuned — BIPIA (indirect),0.999100,0.999101,0.998203,1.000000,0.999996,0,9
ModernBERT fine-tuned — combined,0.984877,0.984772,0.981601,0.987963,0.997750,78,120


## 9. Save consolidated results

In [13]:
consolidated_results = {
    "phase1_original_all_models": phase1_df.reset_index().to_dict(orient="records"),
    "phase2_test_3_models": phase2_test_df.reset_index().to_dict(orient="records"),
    "phase2_inference_latency": phase2_time_df.reset_index().to_dict(orient="records"),
    "phase3_modernbert_deep_dive": phase3_df.reset_index().to_dict(orient="records"),
}

consolidated_json_path = os.path.join(EVAL_RESULTS_DIR, "consolidated_results_v2.json")
with open(consolidated_json_path, "w") as f:
    json.dump(consolidated_results, f, indent=2)

print(f"Saved consolidated results to {consolidated_json_path}")

# saved each phase as a standalone CSV the paper
phase1_df.to_csv(os.path.join(EVAL_RESULTS_DIR, "phase1_original_all_models.csv"))
phase2_test_df.to_csv(os.path.join(EVAL_RESULTS_DIR, "phase2_test_3_models.csv"))
phase2_time_df.to_csv(os.path.join(EVAL_RESULTS_DIR, "phase2_inference_latency.csv"))
phase3_df.to_csv(os.path.join(EVAL_RESULTS_DIR, "phase3_modernbert_deep_dive.csv"))

print("Saved standalone CSVs for each phase.")


Saved consolidated results to /content/drive/MyDrive/Capstone/eval/results/consolidated_results_v2.json
Saved standalone CSVs for each phase.
